In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))

from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT
from SIDER_dataset.libraries.utils import *
%load_ext autoreload
%autoreload 2

In [4]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
paths

12 datasets


[{'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv',
  'dataset_name': 'CPI+fingerprint_cardiac',
  'label_set': ['se_C0016382',
   'se_C0018799',
   'se_C0003811',
   'se_C0428977',
   'se_C0027051',
   'se_C0018790'],
  'features': 2147},
 {'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_high_freq.csv',
  'dataset_name': 'CPI+fingerprint_high_freq',
  'label_set': ['se_C0027497',
   'se_C0018681',
   'se_C0011603',
   'se_C0015230',
   'se_C0042963',
   'se_C0012833'],
  'features': 2147},
 {'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_low_freq.csv',
  'dataset_name': 'CPI+fingerprint_low_freq',
  'label_set': ['se_C0020580',
   'se_C0267792',
   'se_C0018524',
   'se_C0031117',
   'se_C0035078',
   'se_C0011570'],
  'features': 2147},
 {'dataset

In [5]:
k = 10
random_state = 42
performances = []
ranking_criteria = ["MDI"]
include_original_features_options = [True, False]
training_algorithm = "PCT"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]
max_size = 5
cv_results = []

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["dataset_path"]}' training_algorithm:'{training_algorithm}' eval_criterion:'baseline' max_size:'{max_size}' ---"
    print(run_config)
    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    # features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + ["dataset_path"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])

        XofN_groupings, gen_XofN_time = generate_XofN_list_baseline(feature_rankings, 5)
        for include_original_features in include_original_features_options:
            current_train_dataset = group_features(
                train_dataset,
                path["label_set"],
                XofN_groupings,
                include_original_features,
                verbose=True
            )

            current_test_dataset = group_features(
                test_dataset,
                path["label_set"],
                XofN_groupings,
                include_original_features,
                verbose=True
            )

            current_train_dataset.to_csv(f"XofN_naive/tmp/train_dataset.csv", index=False)
            current_test_dataset.to_csv(f"XofN_naive/tmp/test_dataset.csv", index=False)

            original_res, pruned_res, training_time = run_PCT(clus_path,
                                                              "XofN_naive/tmp/train_dataset.csv",
                                                              path["label_set"],
                                                              eval_criteria,
                                                              test_dataset_path=f"XofN_naive/tmp/test_dataset.csv")

            pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                  XofN_groupings,
                                                  gen_XofN_time,
                                                  training_time, path["dataset_name"])
            performances.append(pruned_performance)
            performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                           XofN_groupings,
                                           gen_XofN_time,
                                           training_time, path["dataset_name"])
            performances.append(performance)
    final_perf_df = pd.DataFrame(performances)
    averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
        eval_criteria + ['nodes', 'leaves', 'groups', 'avg_group_features', 'gen_XofN_time',
                         'training_time']].mean().reset_index()
    print(averages)
    cv_results.append(averages)
    performances = []
# laptop 13m
# desktop 9m


--- Running with label:'C:\Users\Voror\Projects\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv' training_algorithm:'PCT' eval_criterion:'baseline' max_size:'5' ---

Fold 1/10 (CPI+fingerprint_cardiac 1/12)
pruning: True, include_original_features: with_org, averageAUROC: 0.6229117, HammingLoss: 0.20144, SubsetAccuracy: 0.47482, RankingLoss: 0.19446, MacroPrecision: 0.64374, MacroRecall: 0.22424, MacroFOne: 0.33084, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6248091, HammingLoss: 0.28297, SubsetAccuracy: 0.28777, RankingLoss: 0.18717, MacroPrecision: 0.39528, MacroRecall: 0.41606, MacroFOne: 0.40002, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5931386, HammingLoss: 0.22182, SubsetAccuracy: 0.45324, RankingLoss: 0.21087, MacroPrecision: 0.54059, MacroRecall: 0.24885, MacroFOne: 0.33774, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5987863, HammingLoss: 0.36571, SubsetAccu

In [6]:
# All results
save_path = "XofN_naive/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,True,with_org,fingerprint_mid_freq,0.565329,0.390004,0.233632,0.274590,0.462715,0.348512,0.394423,175.4,88.2,108.0,5.000000,0.000026,1.307341
1,True,no_org,fingerprint_mid_freq,0.574810,0.389586,0.213198,0.275866,0.462816,0.347901,0.393424,169.0,85.0,108.0,5.000000,0.000026,0.686397
2,False,with_org,fingerprint_mid_freq,0.567078,0.427932,0.125169,0.264522,0.431031,0.502130,0.462080,870.4,435.7,108.0,5.000000,0.000026,1.307341
3,False,no_org,fingerprint_mid_freq,0.569979,0.425347,0.124434,0.260294,0.435645,0.507133,0.466996,882.0,441.5,108.0,5.000000,0.000026,0.686397
4,True,with_org,fingerprint_low_freq,0.510883,0.253590,0.361443,0.296178,0.113877,0.033433,0.043021,15.4,8.2,108.0,5.000000,0.000023,1.302823
5,True,no_org,fingerprint_low_freq,0.502013,0.243724,0.374333,0.301604,0.036667,0.007256,0.000000,4.8,2.9,108.0,5.000000,0.000023,0.686866
6,False,with_org,fingerprint_low_freq,0.595268,0.345858,0.182223,0.248633,0.336207,0.429188,0.372996,828.6,414.8,108.0,5.000000,0.000023,1.302823
7,False,no_org,fingerprint_low_freq,0.581223,0.347992,0.189115,0.260934,0.331956,0.415886,0.364483,848.6,424.8,108.0,5.000000,0.000023,0.686866
8,True,no_org,fingerprint_high_freq,0.553838,0.227580,0.512925,0.165469,0.791260,0.962870,0.867806,36.8,18.9,108.0,5.000000,0.000023,0.656762
9,False,no_org,fingerprint_high_freq,0.581023,0.290390,0.340429,0.162869,0.802653,0.831117,0.815813,607.0,304.0,108.0,5.000000,0.000023,0.656762


In [7]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_naive/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, eval_criteria, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
1,no_org,fingerprint_mid_freq,0.575,0.390,0.213,0.276,0.463,0.348,0.393,169.0; 85.0,108.0; 5.0,0.0,0.0; 0.7
0,with_org,fingerprint_mid_freq,0.565,0.390,0.234,0.275,0.463,0.349,0.394,175.4; 88.2,108.0; 5.0,0.0,0.0; 1.3
5,no_org,fingerprint_low_freq,0.502,0.244,0.374,0.302,0.037,0.007,0.000,4.8; 2.9,108.0; 5.0,0.0,0.0; 0.7
4,with_org,fingerprint_low_freq,0.511,0.254,0.361,0.296,0.114,0.033,0.043,15.4; 8.2,108.0; 5.0,0.0,0.0; 1.3
8,no_org,fingerprint_high_freq,0.554,0.228,0.513,0.165,0.791,0.963,0.868,36.8; 18.9,108.0; 5.0,0.0,0.0; 0.7
11,with_org,fingerprint_high_freq,0.554,0.230,0.507,0.167,0.793,0.955,0.866,51.8; 26.4,108.0; 5.0,0.0,0.0; 1.3
13,no_org,fingerprint_cardiac,0.517,0.260,0.377,0.230,0.118,0.028,0.020,11.8; 6.4,108.0; 5.0,0.0,0.0; 0.7
12,with_org,fingerprint_cardiac,0.557,0.261,0.370,0.227,0.374,0.097,0.115,37.0; 19.0,108.0; 5.0,0.0,0.0; 1.3
17,no_org,CPI_mid_freq,0.596,0.375,0.242,0.276,0.527,0.376,0.432,114.2; 57.6,322.0; 5.0,0.0,0.0; 0.9
16,with_org,CPI_mid_freq,0.606,0.368,0.242,0.274,0.537,0.374,0.435,128.4; 64.7,322.0; 5.0,0.0,0.0; 2.6
